# BERSEKA AI — YOLOv8 Training Pipeline (Kaggle GPU)
Auto-generated oleh `src/training/push_kaggle_kernel.py` — JANGAN edit manual di sini,
edit source di repo `berseka-ai/src/` lalu regenerasi & push ulang.

Mode saat generate: **dry_run**. Ganti env var `NOTEBOOK_MODE` di Kaggle kernel settings
(`dry_run` / `full_run`) tanpa perlu push ulang notebook untuk switch mode.

Target metrik (Backlog 1): mAP@0.5 >= 0.85, akurasi >= 90%, F1 >= 0.85, gap val-train loss <= 15%.

In [ ]:
# Cek GPU yang tersedia (Kaggle biasanya kasih pilihan T4 x2 atau P100)
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# Install dependencies (kernel Kaggle biasanya sudah punya ultralytics, dipin ulang agar konsisten)
!pip install -q ultralytics==8.2.103 albumentations opencv-python-headless pyyaml scikit-learn

In [ ]:
import os, sys, shutil
from pathlib import Path

NOTEBOOK_MODE = os.environ.get("NOTEBOOK_MODE", "dry_run")  # dry_run | full_run
print("NOTEBOOK_MODE =", NOTEBOOK_MODE)

# Repo source code di-attach sebagai Kaggle Dataset (berseka-src) agar tidak perlu
# clone git di setiap run (hemat kuota API/waktu). Lihat kernel-metadata.json dataset_sources.
REPO_SRC = Path("/kaggle/input/berseka-src")
if REPO_SRC.exists():
    sys.path.insert(0, str(REPO_SRC))
    print("Source code ditemukan di", REPO_SRC)
else:
    print("[warn] dataset berseka-src belum di-attach, sebagian import bisa gagal.")

In [ ]:
# --- 1) Download & gabungkan dataset ---
from src.preprocessing.dataset_acquisition import download_all

DATA_ROOT = Path("/kaggle/working/data")
only = None
if NOTEBOOK_MODE == "dry_run":
    # dry-run: cukup dataset kecil dulu untuk validasi pipeline (hemat waktu/kuota)
    only = ["waste_segregation_aashidutt3", "taco_yolo"]

downloaded = download_all(DATA_ROOT, only=only)
print(downloaded)

In [ ]:
# --- 2) Terapkan label mapping (configs/label_mapping.yaml) ke TACO ---
from src.preprocessing.remap_taco_labels import remap_taco_dataset

taco_src = DATA_ROOT / "raw" / "taco_yolo"
taco_remapped = DATA_ROOT / "processed" / "taco_binary"
if taco_src.exists():
    summary = remap_taco_dataset(taco_src, taco_remapped)
    print(summary)
else:
    print("[skip] TACO YOLO belum didownload di mode ini")

In [ ]:
# --- 3) Preprocessing: split train/val/test group-aware + stratified + anti-leakage ---
from src.preprocessing.split_dataset import (
    SampleRecord, default_group_id, group_aware_stratified_split, write_split_manifest, dedup_by_phash
)
import yaml

records = []
# Kumpulkan sample dari TACO ter-remap (punya bbox -> dataset_source='taco')
taco_labels_dir = taco_remapped / "train" / "labels"
if taco_remapped.exists() and taco_labels_dir.exists():
    for lbl_path in taco_labels_dir.glob("*.txt"):
        lines = lbl_path.read_text().strip().splitlines()
        if not lines:
            continue
        cls_id = int(lines[0].split()[0])
        label = "ORGANIC" if cls_id == 0 else "NON_ORGANIC"
        img_path = str(lbl_path).replace("labels", "images").replace(".txt", ".jpg")
        records.append(SampleRecord(path=img_path, label=label, dataset_source="taco",
                                     group_id=default_group_id(img_path)))

print(f"Total sample terkumpul (mode={NOTEBOOK_MODE}): {len(records)}")

if records:
    # dedup pHash HANYA dijalankan di full_run (expensive utk banyak gambar; dry-run skip)
    if NOTEBOOK_MODE == "full_run":
        records, dropped = dedup_by_phash(records, threshold=5)
        print(f"Dedup pHash: {len(dropped)} near-duplicate dibuang")

    splits = group_aware_stratified_split(records, seed=42)
    write_split_manifest(splits, DATA_ROOT / "splits", seed=42)
else:
    print("[skip] tidak ada sample untuk displit pada mode ini")

In [ ]:
# --- 4) Augmentasi field-condition (motion blur, brightness jitter, dll) ---
from src.preprocessing.augmentation import build_field_condition_transform

transform = build_field_condition_transform()
print(f"Pipeline augmentasi siap: {len(transform.transforms)} operasi "
      "(motion blur, gaussian blur/noise, jpeg artifact, brightness/contrast, occlusion, hue shift)")
# Augmentasi mosaic/mixup/hsv bawaan YOLOv8 diterapkan otomatis saat model.train() (lihat sel training)

In [ ]:
# --- 5) Training YOLOv8 ---
from src.training.train import train as run_training

data_yaml = str(taco_remapped / "data.yaml") if taco_remapped.exists() else None
if data_yaml:
    result = run_training(mode=NOTEBOOK_MODE, data_yaml=data_yaml, resume=True)
    print(result)
else:
    print("[skip] data.yaml belum ada, lewati training (mode verifikasi struktur saja)")

In [ ]:
# --- Simpan checkpoint ke /kaggle/working agar bisa di-attach sbg Kaggle Dataset
# 'berseka-checkpoints' untuk RESUME di run berikutnya (hemat kuota GPU, tidak dari nol).
import glob, shutil
ckpt_dir = Path("/kaggle/working/checkpoints")
best_ckpts = glob.glob(str(ckpt_dir / "**" / "weights" / "last.pt"), recursive=True)
print("Checkpoint ditemukan:", best_ckpts)
# Setelah run ini selesai: buat/update Kaggle Dataset 'berseka-checkpoints' dari
# /kaggle/working/checkpoints via `kaggle datasets version` (dilakukan manual/CI terpisah
# supaya checkpoint tidak hilang saat kernel session berakhir).

In [ ]:
# --- 6) Evaluasi metrik otomatis (mAP, precision, recall, F1, confusion matrix, loss curve) ---
from src.training.evaluate import evaluate

if best_ckpts:
    weights = best_ckpts[0].replace("last.pt", "best.pt")
    results_csv = str(Path(best_ckpts[0]).parent.parent / "results.csv")
    report = evaluate(weights=weights, data_yaml=data_yaml, results_csv=results_csv,
                       out_dir="/kaggle/working/eval_output")
else:
    print("[skip] tidak ada checkpoint untuk dievaluasi pada mode ini")